# Fase 1 · Paso 4: cierre de la revisión de `tibiofemoral_crop_v0.4_pilot`

Esta libreta valida el registro visual ciego ya completado y genera el resumen público de la revisión de `v0.4`. La revisión no congela parámetros porque contiene rechazos. No procesa la cohorte completa, no crea particiones, no entrena y no abre la prueba reservada.

## 1. Montar Drive y localizar la revisión privada

In [ ]:
import os
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
private_root = userdata.get('KNEE_DATA_ROOT') or os.environ.get('KNEE_DATA_ROOT')
if not private_root:
    raise RuntimeError('Configura el secreto KNEE_DATA_ROOT antes de continuar.')
os.environ['KNEE_DATA_ROOT'] = private_root
private_root_path = Path(private_root)
output_dir = private_root_path / 'outputs' / 'auditorias' / 'fase_1' / 'paso_4_localizacion_tibiofemoral' / 'v0_4_piloto'
review_path = output_dir / 'revision_visual_ciega.csv'
if not review_path.is_file():
    raise FileNotFoundError('No se encontró la revisión visual ciega de v0.4.')
print('Drive montado y revisión privada de v0.4 localizada.')

## 2. Obtener una revisión identificable del repositorio

In [ ]:
import subprocess

REPO_URL = 'https://github.com/AlonsoYess/Knee.git'
BRANCH = 'codex/fase-0-base-reproducible'
REPO_DIR = Path('/content/Knee_fase1_paso4_cierre_v04')

if REPO_DIR.exists():
    if not (REPO_DIR / '.git').is_dir():
        raise RuntimeError('La ruta de trabajo existe y no es un clon Git; usa un entorno nuevo.')
    dirty = subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout
    if dirty:
        raise RuntimeError('El clon temporal contiene cambios. Restablece el entorno de ejecución.')
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'merge', '--ff-only', f'origin/{BRANCH}'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
print('Revisión preparada:', commit)

## 3. Instalar y comprobar el código

In [ ]:
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=REPO_DIR, check=True)
if subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout:
    raise RuntimeError('Las comprobaciones modificaron el repositorio temporal.')
print('Código y pruebas automáticas preparados.')

## 4. Validar la revisión ciega y generar el resumen público

In [ ]:
subprocess.run(
    [
        sys.executable,
        '-m',
        'knee.joint_review',
        '--output-dir',
        str(output_dir),
        '--algorithm-version',
        'tibiofemoral_crop_v0.4_pilot',
        '--expected-knees',
        '20',
    ],
    cwd=REPO_DIR,
    check=True,
)

## 5. Registrar el resultado controlado

In [ ]:
import json
from datetime import datetime, timezone

summary_path = output_dir / 'resumen_revision_visual_publico.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
if summary.get('status') != 'rejected_after_blinded_review':
    raise RuntimeError('El resultado de v0.4 no coincide con la revisión registrada.')
if summary.get('reviewed_knees') != 20 or summary.get('acceptable_crops') != 8 or summary.get('rejected_crops') != 12:
    raise RuntimeError('El resumen de la revisión de v0.4 contiene conteos inesperados.')
if summary.get('technical_exclusions') != 0 or not summary.get('reviewed_without_outcome'):
    raise RuntimeError('La revisión de v0.4 no conserva el cegamiento o registra exclusiones inesperadas.')
expected_reasons = {'joint_not_centered': 9, 'peripheral_artifact_present': 6}
if summary.get('rejection_reason_counts') != expected_reasons:
    raise RuntimeError('Los motivos agregados de rechazo de v0.4 no coinciden con la revisión ciega.')
if summary.get('parameters_frozen'):
    raise RuntimeError('Los parámetros de una versión rechazada no pueden quedar congelados.')
record = {
    'phase': 1,
    'step': 4,
    'operation': 'close_blinded_tibiofemoral_localization_v04_review',
    'executed_at_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': commit,
    'status': summary['status'],
    'algorithm_version': summary['algorithm_version'],
    'reviewed_knees': summary['reviewed_knees'],
    'acceptable_crops': summary['acceptable_crops'],
    'rejected_crops': summary['rejected_crops'],
    'technical_exclusions': summary['technical_exclusions'],
    'rejection_reason_counts': summary['rejection_reason_counts'],
    'parameters_frozen': summary['parameters_frozen'],
    'mass_processing_executed': False,
    'partitions_created': False,
    'training_executed': False,
    'reserved_test_opened': False,
    'review_provenance': 'Nueva revisión técnica asistida por Codex el 2026-10-01 sobre las 20 imágenes facilitadas por el investigador. No es transcripción de la revisión histórica; se conocía el agregado 8/20 y no se consultaron desenlaces individuales.',
    'closure_execution_environment': 'Google Colab, libreta 12',
}
(output_dir / 'registro_cierre_revision.json').write_text(
    json.dumps(record, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)
print('v0.4 rechazada tras revisión visual ciega: 8/20 recortes aceptables. El paso 4 continúa abierto y las etapas posteriores siguen bloqueadas.', record)